# 02 — Augmentation Pipeline

1500 satirlik egitim setini uretir, dagilimlari gorsellestirir ve sanity check yapar.

**Cikti:** `data/processed/training_set_v1.parquet`

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

SEED = 42
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
from ml.src.augmentation import build_training_set

df = build_training_set(target_rows=1500, seed=SEED)
print('shape:', df.shape)
df.head(3).T

In [ ]:
out_path = REPO_ROOT / 'data' / 'processed' / 'training_set_v1.parquet'
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(out_path, index=False)
print('saved:', out_path)

## Material distribution (rapor §5.5)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['raw_material'].value_counts().plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_title('Raw material distribution (target 0.70 / 0.20 / 0.10)')
ax.set_ylabel('count')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df['processing_route_candidate'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Processing route candidate distribution (15 classes)')
ax.set_ylabel('count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df['expected_profit_try'].clip(lower=df['expected_profit_try'].quantile(0.01),
                                  upper=df['expected_profit_try'].quantile(0.99)).hist(bins=60, ax=ax, color='#2ca02c')
ax.set_title('expected_profit_try (1-99 percentile clipped)')
ax.set_xlabel('TRY')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for mode, color in zip(['kara','deniz','demiryolu','hava'], ['#1f77b4','#ff7f0e','#2ca02c','#d62728']):
    sub = df[df['transport_mode']==mode]
    ax.scatter(sub['total_distance_km'], sub['co2_kg'], s=8, alpha=0.4, label=mode, color=color)
ax.set_xlabel('total_distance_km')
ax.set_ylabel('co2_kg (log)')
ax.set_yscale('log')
ax.set_title('co2_kg vs total_distance_km, colored by transport_mode')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
df.describe().T

In [ ]:
df.isna().sum().sort_values(ascending=False).head(15)

## Feature pipeline fit (Adim 4)

ColumnTransformer (one-hot + median impute) fit edilip `models/feature_pipeline.pkl`'a kaydedilir.

In [ ]:
from ml.src.feature_pipeline import fit_and_save
from ml.src.features import M1_FEATURES
import joblib

models_dir = REPO_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
out_pkl = models_dir / 'feature_pipeline.pkl'

pipe = fit_and_save(df, kind='xgboost', out_path=str(out_pkl))
print('saved:', out_pkl)

X = pipe.transform(df.head(3)[M1_FEATURES])
print('transformed shape:', X.shape)
print('feature_names_out (count):', len(pipe.get_feature_names_out()))
print('first 10:', list(pipe.get_feature_names_out())[:10])